# DAPPC LAB 3 - Identification of relevant features for subgroups of patients

This notebook guides you in the implementation of the Ant-Colony Optimization (ACO) algorithm to perform Feature Selection (FS).

## Recommended workflow
1. Load the dataset obtained after the previous laboratory steps.
2. Identify the most populated SOM cluster containing samples from all outcome classes of interest.
3. Initialize the main paramters of the ACO algorithm.
4. Run ACO-based feature selection.
5. Analyze convergence, the selected feature subset, and final predictive performance.


## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
import ACO
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams['figure.figsize'] = (9, 4)


## 1. Load data

Load the four sets of clusters obtained at the end of LAB2: the best set of clusters of each of the four datasets (each SOM)

In [ ]:
file_path = r"C:\Users\alexi\OneDrive - Politecnico di Torino\Desktop\Politecnico\Didattica\2025-2026\DAPPC 2026\LAB3\cluster_lab3.xlsx"

cluster_df = pd.read_excel(file_path)
# ============================================================
# Load files
# ============================================================


## 2. Select the largest valid SOM cluster

We will perform feature selection (FS) with Ant-Colony Optimization (ACO) algorithm only on two clusters. Identify them as follows:
1. Identify the best set of cluster as the one with minimum value of maximum intra-cluster variability: you choose the best SOM.
2. Remove the clusters where all the three classes are not present.
3. Remove the clusters that are too small (it depends on the distribution of the patients within the different clusters).
4. Identify the two more distant clusters (inter-cluster distances computed on the centroids).


In [ ]:
...

## 3. Prepare data for ACO

At this point we restrict the analysis to the selected clusters: perform the ACO for the first one and then repeat it for the other one.
As always, identify only the input featuers (not the IDs)


In [ ]:
cluster_df = df[df[cluster_col] == selected_cluster].copy()
print('Selected cluster shape:', cluster_df.shape)

# Extract input features
feature_cols = ...
X_df = ...
n_features = ... # identify the number of features
# Extract the output column
y = ...

print('Number of candidate features:', X_df.shape[1])
print('Class distribution within selected cluster:')
display(pd.Series(y).value_counts().sort_index())

## 4. Scale the features

Only the predictors are normalized. The target is kept unchanged.
For simplicity, we apply Min-Max scaling before the filter and wrapper stages.

In [ ]:
scaler = MinMaxScaler()
X = scaler.fit_transform(X_df)
print('Scaled feature matrix shape:', X.shape)


## 5. ACO feature selection

The algorithm used in this lab has two key characteristics:

1. the subset size is **not fixed a-priori**;
2. the stop rule is **adaptive** and depends on the recent marginal improvement of the kNN-based objective function.

### Objective function

In this implementation, the ACO procedure is formulated as a **minimization problem**.

For each candidate subset of features $S$, we define the following objective function:

$$
J(S) = \lambda \, \bigl(1 - \mathrm{BalancedAccuracy}_{CV}(S)\bigr)
      + (1 - \lambda)\,\frac{|S|}{p}
$$

where:

- $\mathrm{BalancedAccuracy}_{CV}(S)$ is the cross-validated balanced accuracy obtained by a kNN classifier trained on the selected subset $S$;
- $|S|$ is the number of selected features;
- $p$ is the total number of available features;
- $\lambda \in [0,1]$ controls the trade-off between predictive performance and subset compactness.

The first term represents the **classification error**, while the second term represents the **relative size of the subset**.

Therefore, the algorithm searches for the subset that minimizes a weighted combination of:

1. prediction error,
2. subset cardinality.

Note: the entire algorithm is implemented in the **ACO.py** file.

## 6. Set the ACO parameters

The default values below remain close to the historical lab when possible, while introducing the adaptive stopping mechanism.

We evaluated three different ACO parameter setups in order to compare how different search conditions influence the feature-selection process.

### Common parameters

The following parameters are shared by all three setups:

- `n_ants = 5`
- `alpha = 1.0`
- `q_constant = 1.0`
- `cv_splits = 5`
- `scoring = "balanced_accuracy"`
- `stop_prob_if_tiny = 0.25`
- `stop_prob_if_small = 0.10`
- `small_delta_threshold = 0.002`
- `medium_delta_threshold = 0.010`
- `stop_prob_if_nonpos = 0.60`
- `max_features = 10`

### Parameters that vary across setups

The three configurations differ in the following parameters:

- `iterations`: controls the maximum length of the search process;
- `beta`: controls the influence of the heuristic information during feature selection;
- `initial_pheromone`: defines the initial pheromone level assigned to the features;
- `evaporation_rate`: regulates how quickly pheromone information fades over time;
- `min_features`: defines the minimum number of features that must be selected before stopping is allowed;
- `lambda_score`: controls the trade-off between predictive performance and subset compactness;
- `delta_window`: determines how many recent improvements are considered by the adaptive stopping rule.

### Influence of the varying parameters

- **`iterations`**  
  A higher number of iterations allows the colony to explore the search space for a longer time.  
  A lower number of iterations makes the search shorter and more constrained.

- **`beta`**  
  This parameter controls how strongly the ants rely on the initial heuristic information.  
  Higher values of `beta` increase the importance of the a-priori feature relevance.

- **`initial_pheromone`**  
  This parameter defines the initial pheromone assigned to all features.  
  It affects the starting balance between exploration and reinforcement.

- **`evaporation_rate`**  
  This parameter controls how quickly pheromone information is reduced at each iteration.  
  A higher evaporation rate reduces the influence of previous iterations more quickly.

- **`min_features`**  
  This parameter imposes a minimum subset size before the algorithm is allowed to stop.  
  It directly affects how early the adaptive stopping rule can intervene.

- **`lambda_score`**  
  This parameter balances predictive performance and subset size in the objective function.  
  Higher values give more importance to classification performance, while lower values increase the role of compactness.

- **`delta_window`**  
  This parameter determines how many recent improvements are averaged when evaluating the adaptive stopping condition.  
  It affects how sensitive the stopping mechanism is to short-term changes in the objective function.

### Setup A

- `iterations = 300`
- `beta = 2.0`
- `initial_pheromone = 0.25`
- `evaporation_rate = 0.10`
- `min_features = 4`
- `max_features = 10`
- `lambda_score = 0.90`
- `delta_window = 4`

### Setup B

- `iterations = 300`
- `beta = 2.0`
- `initial_pheromone = 0.50`
- `evaporation_rate = 0.10`
- `min_features = 4`
- `max_features = 10`
- `lambda_score = 0.94`
- `delta_window = 3`

### Setup C

- `iterations = 150`
- `beta = 3.0`
- `initial_pheromone = 0.50`
- `evaporation_rate = 0.10`
- `min_features = 4`
- `max_features = 10`
- `lambda_score = 0.90`
- `delta_window = 3`

In [ ]:
n_ants = ...
iterations = ...         
alpha = ...
beta = ...
initial_pheromone = ...
evaporation_rate = ...
q_constant = ...
n_features = ...
min_features = ...
max_features = ...
lambda_score = ...
delta_window = ...
scoring = ...
cv_splits = ...

heuristic_weights = np.ones(n_features) / n_features

## 7. Run adaptive ACO

In [ ]:
selector = ACO.AdaptiveACOFeatureSelector(
    X=X,
    y=y,
    feature_names=feature_cols,
    heuristic_weights=heuristic_weights,
    estimator=KNeighborsClassifier(n_neighbors=5),
    n_ants=n_ants,
    iterations=iterations,
    alpha=alpha,
    beta=beta,
    initial_pheromone=initial_pheromone,
    evaporation_rate=evaporation_rate,
    q_constant=q_constant,
    min_features=min_features,
    max_features=max_features,
    lambda_score=lambda_score,
    delta_window=delta_window,
    scoring=scoring,
    cv_splits=cv_splits,
    random_state=RANDOM_STATE,
)

selector.fit()
print('Best global objective function:', selector.best_cost_)
print('Best global CV score:', selector.best_cv_score_)
print('Number of selected features:', len(selector.get_selected_indices()))


## 8. Convergence analysis

In [ ]:
plt.figure()
plt.plot(selector.curve_)
plt.xlabel('Iteration')
plt.ylabel('Best objective value so far')
plt.title('Adaptive ACO convergence curve')
plt.tight_layout()
plt.show()

display(selector.history_.head()) #you can save the entire history to see how it evolves
display(selector.history_.tail())


## 9. Final selected feature subset

Remember to save all the results obtained.

In [ ]:
selected_idx = selector.get_selected_indices()
selected_features = selector.get_selected_feature_names()

selected_df = pd.DataFrame({
    'selected_rank': np.arange(1, len(selected_features) + 1),
    'feature': selected_features,
    'initial_score': ranking_df.set_index('feature').loc[selected_features, 'score'].values,
})

display(selected_df)

# save here the results (you can save in the kind of file you prefer)